# Understanding the previous group's Kalman filter
An annotated walkthrough for the Sensing and Sensor Technology project.

This notebook explains the **original linear Kalman filter**. Its behavior is preserved, including the heading-wraparound limitation. It is not yet an EKF. Run the code cells from top to bottom; the complete class is kept in one cell because Python class definitions cannot be continued across ordinary notebook cells.

We use local position in metres, time in seconds, and assume heading is supplied in radians. Check the real sensor's units and heading convention before using its readings.

If imports fail, run `%pip install numpy filterpy` in a separate code cell in this notebook's kernel.

## 1. What are we estimating?
The state is
$$
\mathbf{x}=[x,\ y,\ \psi,\ v_x,\ v_y,\ \omega]^T.
$$
Here $\psi$ is yaw and $\omega$ is yaw rate. GPS measures position; the IMU heading input measures yaw. Velocity and yaw rate are estimated indirectly in this code.

`tracker = kal_fil()` creates our wrapper object. `tracker.kf` is the FilterPy object inside it. `tracker.kf.x` is the full estimated state vector; its first entry is the position coordinate $x$. No filter is being called twice.

In [1]:
import numpy as np

class kal_fil:
    """Original linear position/heading filter, annotated for study."""

    def __init__(self):
        from filterpy.kalman import KalmanFilter

        # Six estimated variables: [x, y, yaw, vx, vy, yaw_rate].
        # Measurement dimension is changed for each sensor update below.
        self.kf = KalmanFilter(dim_x=6, dim_z=3)

        # Identity: retain each state entry. predict() adds motion terms.
        self.kf.F = np.eye(6)

        # Map the six-state vector to the two predicted GPS coordinates.
        self.H_pos = np.array([
            [1, 0, 0, 0, 0, 0],
            [0, 1, 0, 0, 0, 0]
        ])

        # Map the state vector to the predicted yaw measurement.
        self.H_yaw = np.array([
            [0, 0, 1, 0, 0, 0]
        ])

        # Initial covariance: FilterPy starts P as identity, so P = 5 I.
        # Diagonal entries are variances; initial cross-covariances are zero.
        self.kf.P *= 5

        # Original process-noise covariance, added once per prediction.
        # Limitation: this stays fixed even when dt changes.
        self.kf.Q = np.eye(6) * 0.05

        # Measurement VARIANCES, not standard deviations.
        self.R_pos = np.diag([0.5, 0.5])  # m^2, assuming local metre coordinates
        self.R_yaw = np.array([[0.05]])   # rad^2, assuming yaw is in radians

    def predict(self, dt):
        """Predict state and uncertainty over dt seconds."""
        self.kf.F[0, 3] = dt  # x_next = x + vx * dt
        self.kf.F[1, 4] = dt  # y_next = y + vy * dt
        self.kf.F[2, 5] = dt  # yaw_next = yaw + yaw_rate * dt

        # Computes x_prior = F @ x and P_prior = F @ P @ F.T + Q.
        self.kf.predict()

    def update_heading(self, yaw):
        """Correct the estimate using a NEW heading sample."""
        self.kf.dim_z = 1

        # Uses ordinary subtraction for the residual.
        # WARNING: the original code does NOT wrap the angle residual.
        self.kf.update(np.array([yaw]), R=self.R_yaw, H=self.H_yaw)

    def update_position(self, pos_x, pos_y):
        """Correct the estimate using a NEW GPS position sample."""
        z = np.array([pos_x, pos_y])
        self.kf.dim_z = 2
        self.kf.update(z, R=self.R_pos, H=self.H_pos)

## 2. Prediction: the physical model inside F
The three off-diagonal entries give
$$
F=\begin{bmatrix}
1&0&0&\Delta t&0&0\\
0&1&0&0&\Delta t&0\\
0&0&1&0&0&\Delta t\\
0&0&0&1&0&0\\
0&0&0&0&1&0\\
0&0&0&0&0&1
\end{bmatrix}.
$$
The prediction equations are
$$
\hat{\mathbf{x}}_{k|k-1}=F\hat{\mathbf{x}}_{k-1|k-1},\qquad
P_{k|k-1}=FP_{k-1|k-1}F^T+Q.
$$
The notation $k|k-1$ means “at time k, using measurements through time k−1.”

This assumes constant $v_x$, $v_y$, and $\omega$ during prediction. The state prediction does not use acceleration measurements or heading to rotate velocity. There is no control input $Bu$ in this implementation.

Putting `2*dt` in `F[0, 3]` doubles the contribution of $v_x$ to predicted displacement; it changes the physical model. The actual constant-velocity relation uses `dt`.

In [2]:
# A transparent numerical example, independent of real hardware.
tracker = kal_fil()
tracker.kf.x = np.array([10., 20., 0.3, 2., 1., 0.]).reshape(6, 1)
tracker.predict(dt=1.0)
print("Predicted state:", tracker.kf.x.ravel())
# Expected: [12., 21., 0.3, 2., 1., 0.]

Predicted state: [12.  21.   0.3  2.   1.   0. ]


## 3. P, Q and R describe different uncertainties
| Matrix | Meaning | Where it enters |
|---|---|---|
| $P$ | Current estimation-error covariance | Propagated in prediction and reduced by informative updates |
| $Q$ | Process-noise covariance over the prediction interval | Added during prediction |
| $R$ | Measurement-noise covariance | Used to weight each sensor update |

A diagonal entry of $P$ is a variance. Off-diagonal entries describe how errors in different state quantities covary. For example, an uncertain velocity creates uncertainty in future position.

`P *= 5` gives each state variable a variance of 5 in its own squared units. This is a rough initial choice, not a measured characterization. FilterPy's default state is zero; a real implementation should choose sensible initial state values and uncertainties.

`Q = 0.05 * I` adds the same numerical variance to each state component per prediction. A physically constructed $Q$ generally depends on $\Delta t$ and may contain off-diagonal terms.

The chosen measurement uncertainties are:
$$
\sigma_x=\sigma_y=\sqrt{0.5}\ \mathrm m\approx0.71\ \mathrm m,
\qquad
\sigma_\psi=\sqrt{0.05}\ \mathrm{rad}\approx12.8^\circ.
$$
These are assumed settings, not established sensor specifications. The zero off-diagonal terms in $R_{\rm pos}$ assume uncorrelated x/y measurement errors.

## 4. H predicts what each sensor should report
The observation matrices select state entries:
$$
H_{\rm pos}\mathbf{x}=\begin{bmatrix}x\\y\end{bmatrix},
\qquad H_{\rm yaw}\mathbf{x}=\psi.
$$
Thus $H_{\rm pos}$ has shape $(2,6)$ and $H_{\rm yaw}$ has shape $(1,6)$. The row count is the number of measurements in that update.

The **innovation** is actual measurement minus predicted measurement:
$$
\mathbf r=\mathbf z-H\hat{\mathbf x}_{k|k-1}.
$$
For predicted position $(10,20)$ and measured position $(12,19)$, the innovation is $(2,-1)$. It is a discrepancy to be weighted, not a command to overwrite the estimate.

In [3]:
example_state = np.array([10., 20., 0.3, 2., 1., 0.])
gps_measurement = np.array([12., 19.])
predicted_measurement = tracker.H_pos @ example_state
innovation = gps_measurement - predicted_measurement
print("Predicted GPS measurement:", predicted_measurement)
print("Position innovation:", innovation)

Predicted GPS measurement: [10. 20.]
Position innovation: [ 2. -1.]


## 5. What update() calculates
For either sensor, the linear update is
$$
S=HP^-H^T+R,\qquad K=P^-H^TS^{-1},
$$
$$
\hat{\mathbf x}^{+}=\hat{\mathbf x}^{-}+K\mathbf r.
$$
Here minus means predicted and plus means updated. $S$ is the innovation covariance and $K$ is the Kalman gain. For GPS, $K$ is $(6,2)$; for heading it is $(6,1)$.

In exact arithmetic the covariance update can be written
$$
P^+=(I-KH)P^-.
$$
A numerically robust equivalent is the Joseph form:
$$
P^+=(I-KH)P^-(I-KH)^T+KRK^T.
$$
For a scalar directly observed state, $K=P^-/(P^-+R)$. A larger measurement variance gives that measurement less weight. In the general matrix case, cross-covariances also determine how each residual changes the state.

**Why can GPS change velocity?** Prediction creates position–velocity covariance. A position residual can therefore correct velocity through the gain. With the original initialization and matrices, the three pairs $(x,v_x)$, $(y,v_y)$ and $(\psi,\omega)$ remain independent, so yaw updates do not correct position.

Changing `dim_z` only declares how many values are in the current measurement. It does not change the six-state model or turn this into an EKF.

In [4]:
# Continue the tracker from the prediction example.
# GPS is one metre ahead in x and one metre behind in y.
tracker.update_position(pos_x=13.0, pos_y=20.0)
print("After GPS:", tracker.kf.x.ravel())

# This heading is near the current estimate, away from the wrap boundary.
tracker.update_heading(yaw=0.32)
print("After heading:", tracker.kf.x.ravel())

After GPS: [12.95260664 20.04739336  0.3         2.47393365  0.52606635  0.        ]
After heading: [1.29526066e+01 2.00473934e+01 3.19900990e-01 2.47393365e+00
 5.26066351e-01 9.90099010e-03]


## 6. The heading-wraparound problem
The original heading update uses ordinary subtraction. For predicted $179^\circ$ and measured $-179^\circ$, it gets $-358^\circ$, although the shortest signed rotation is $+2^\circ$.

With a yaw gain of 0.5, the unwrapped residual gives $179-179=0^\circ$: the opposite direction. The wrapped residual gives $179+1=180^\circ$.

For radians the correct shortest-angle residual is
$$
r_\psi=((\psi_{\rm measured}-\psi_{\rm predicted}+\pi)\bmod 2\pi)-\pi.
$$
It must be wrapped **before applying the gain**. Wrapping only the final state does not repair an incorrect correction. Both input angles being inside $[-\pi,\pi)$ is also insufficient.

The next cell demonstrates the issue; it does not modify the original class. Repairing that update is a separate implementation exercise.

In [5]:
predicted_yaw = np.deg2rad(179.0)
measured_yaw = np.deg2rad(-179.0)
raw_residual = measured_yaw - predicted_yaw
wrapped_residual = (raw_residual + np.pi) % (2 * np.pi) - np.pi
print("Raw residual (degrees):", np.rad2deg(raw_residual))
print("Wrapped residual (degrees):", np.rad2deg(wrapped_residual))
print("Update with raw residual:", np.rad2deg(predicted_yaw + 0.5 * raw_residual))
print("Update with wrapped residual:", np.rad2deg(predicted_yaw + 0.5 * wrapped_residual))

Raw residual (degrees): -358.0
Wrapped residual (degrees): 1.9999999999999774
Update with raw residual: 0.0
Update with wrapped residual: 180.0


## 7. Using the class with sensor timestamps
Predict once from the current filter time to the next measurement time. Then update only with the newly arrived measurement. If GPS and heading have the same timestamp, predict to that time once and apply both updates without advancing time between them.

Do not repeatedly reuse one old heading sample as if it were new independent evidence. This artificially increases confidence. During a GPS outage, continue prediction and apply fresh heading updates when available; this model still extrapolates position using its estimated constant velocity.

`tracker.kf.x.ravel()` returns a flattened view or copy of the current state. It does not run prediction or correction.

## 8. What changes for an EKF?
This class uses a linear model $\mathbf{x}_{k+1}=F\mathbf{x}_k$. A possible learning prototype instead uses $[x,y,v,\psi,\omega]^T$ and
$$
x_{k+1}=x_k+v_k\cos\psi_k\Delta t,\qquad
y_{k+1}=y_k+v_k\sin\psi_k\Delta t.
$$
That assumes motion along the platform's heading. It couples heading to displacement and is nonlinear. The EKF predicts the state with the nonlinear function $f$, while using its Jacobian to propagate covariance. This remains a simplified prototype, not the full IMU-driven, bias-estimating navigation system.

## Small checks for yourself
1. Which entry of $H_{\rm yaw}$ would change if you reordered the state?
2. If GPS becomes noisier, which matrix should represent that change?
3. Why can a GPS update change $v_x$ when GPS measures only position?
4. Why does increasing $Q$ usually make the next measurement more influential?

Reference documentation: [FilterPy KalmanFilter](https://filterpy.readthedocs.io/en/latest/kalman/KalmanFilter.html). The code being explained was supplied in our conversation.

1. the third element, because $H \mathbf{x}= \psi$
2. should be the R matrices, they are responsible for measurement noise
3. due to the transition matrix F, we can numerically update accordingly to dt, $\hat{\mathbf{x}}_{k|k-1}=F\hat{\mathbf{x}}_{k-1|k-1}$. but this might change in an EKF
4. because Q is teh process noise and $P_{k|k-1}=FP_{k-1|k-1}F^T+Q$ is the current state error, which is contributed by Q

In [6]:
import sympy as sy
x, y, psi, vx, vy, omega, dt = sy.symbols( "x y psi vx vy omega dt", real=True)
state = sy.Matrix([x, y, psi, vx, vy, omega])
v = vx * sy.cos(psi) + vy * sy.sin(psi)
psi_next = psi + omega * dt
motion = sy.Matrix([
    x + v / omega * (sy.sin(psi_next) - sy.sin(psi)),
    y + v / omega * (sy.cos(psi) - sy.cos(psi_next)),
    psi_next,
    v * sy.cos(psi_next),
    v * sy.sin(psi_next),
    omega])

display(motion)
J = motion.jacobian(state)
display(J)

Matrix([
[x + (vx*cos(psi) + vy*sin(psi))*(-sin(psi) + sin(dt*omega + psi))/omega],
[ y + (vx*cos(psi) + vy*sin(psi))*(cos(psi) - cos(dt*omega + psi))/omega],
[                                                         dt*omega + psi],
[                        (vx*cos(psi) + vy*sin(psi))*cos(dt*omega + psi)],
[                        (vx*cos(psi) + vy*sin(psi))*sin(dt*omega + psi)],
[                                                                  omega]])

Matrix([
[1, 0, (-vx*sin(psi) + vy*cos(psi))*(-sin(psi) + sin(dt*omega + psi))/omega + (vx*cos(psi) + vy*sin(psi))*(-cos(psi) + cos(dt*omega + psi))/omega, (-sin(psi) + sin(dt*omega + psi))*cos(psi)/omega, (-sin(psi) + sin(dt*omega + psi))*sin(psi)/omega, dt*(vx*cos(psi) + vy*sin(psi))*cos(dt*omega + psi)/omega - (vx*cos(psi) + vy*sin(psi))*(-sin(psi) + sin(dt*omega + psi))/omega**2],
[0, 1,  (-vx*sin(psi) + vy*cos(psi))*(cos(psi) - cos(dt*omega + psi))/omega + (vx*cos(psi) + vy*sin(psi))*(-sin(psi) + sin(dt*omega + psi))/omega,  (cos(psi) - cos(dt*omega + psi))*cos(psi)/omega,  (cos(psi) - cos(dt*omega + psi))*sin(psi)/omega,  dt*(vx*cos(psi) + vy*sin(psi))*sin(dt*omega + psi)/omega - (vx*cos(psi) + vy*sin(psi))*(cos(psi) - cos(dt*omega + psi))/omega**2],
[0, 0,                                                                                                                                          1,                                                0,                                     

In [9]:
from filterpy.common import Q_discrete_white_noise
from filterpy.kalman import ExtendedKalmanFilter
from trial_EKF import Ekal_fil
from numpy import eye, array, asarray
import numpy as np

rk = Ekal_fil()

# Start at the origin, heading east, moving at 1 m/s.
rk.kf.x = np.array([
    [0.0],  # x
    [0.0],  # y
    [0.0],  # psi
    [1.0],  # vx
    [0.0],  # vy
    [1.0],  # omega
])

rk.predict(0.05)

print(rk.kf.x.ravel())
print(np.isfinite(rk.kf.P).all())

[0.04997917 0.00124974 0.05       0.99875026 0.04997917 1.        ]
True
